# Mirage · dev evaluation (seed 0)

Scores the base model and both seed-0 arms on **Spider dev** (1,034 questions, 20 unseen databases), once.
Nothing here trains or tunes. ~2.5–3 h on a T4.

**Inputs (right panel → Add Input):** `spider` (Your Datasets) **and** `notebook376c8ebfe2` (Your Work, version 2).
Accelerator **GPU T4 x2**, Internet **On**. Then **Save Version → Save & Run All (Commit)**.
Read the result from the version's **Notebook** tab (last cell) or **Output → runs/report/REPORT.md**.

In [ ]:
# --- setup ---------------------------------------------------------------
import os, subprocess, sys, time, json, shutil
T0 = time.time()
SPIDER_DIR = "/kaggle/input/datasets/siddhartharajat/spider/spider_data"
RUNS = "/kaggle/working/runs"
os.environ.update(SPIDER_DIR=SPIDER_DIR, MIRAGE_RUNS=RUNS, CUDA_VISIBLE_DEVICES="0",
                  PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
os.makedirs(RUNS, exist_ok=True)
os.chdir("/kaggle/working")
if not os.path.exists("mirage"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/SiddharthaRaja/Mirage", "mirage"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "mirage[train]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
os.chdir("/kaggle/working/mirage")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
assert os.path.exists(f"{SPIDER_DIR}/dev.json"), "SPIDER_DIR is wrong"
import torch
assert torch.cuda.is_available(), "NO GPU: set Accelerator to GPU T4 x2"
print(torch.cuda.get_device_name(0))

In [ ]:
# --- bring in the trained run (suites + both adapters) -----------------------
hits = []
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    if root.endswith("/runs") and "arms" in dirs:
        hits.append(root)
    if root.count("/") - "/kaggle/input".count("/") > 6:
        dirs[:] = []
print("found runs folders:", hits)
if not hits:
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):   # show what IS attached
        if root.count("/") - "/kaggle/input".count("/") <= 3: print(root, dirs[:8])
    raise SystemExit("notebook376c8ebfe2 output not attached: Add Input -> Your Work -> notebook376c8ebfe2")
src = hits[0]
for part in ["suites", "arms", "filter", "logs", "RUN2.md"]:
    s, d = f"{src}/{part}", f"{RUNS}/{part}"
    if os.path.isdir(s): shutil.copytree(s, d, dirs_exist_ok=True)
    elif os.path.exists(s): shutil.copy(s, d)
for a in ["single", "suite"]:
    ad = f"{RUNS}/arms/{a}/adapter"
    assert os.path.exists(f"{ad}/adapter_config.json"), f"missing adapter: {ad}"
    print(a, "adapter ok:", sorted(os.listdir(ad))[:6])
print("suites:", len(os.listdir(f"{RUNS}/suites")), "databases")

In [ ]:
# --- helpers ----------------------------------------------------------------
def sh(cmd):
    print(f"\n$ {cmd}   [{(time.time()-T0)/3600:.2f} h in]", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end="", flush=True); out.append(line)
    if p.wait(): raise RuntimeError(f"failed: {cmd}")
    return "".join(out)

## Generate on dev (greedy + 8 samples per question), then score on CPU

In [ ]:
MODELS = {"start": None,
          "grpo-single": f"{RUNS}/arms/single/adapter",
          "grpo-suite": f"{RUNS}/arms/suite/adapter"}
for name, adapter in MODELS.items():
    out = f"{RUNS}/eval/{name}-dev"
    a = f"--adapter {adapter} " if adapter else ""
    sh(f"mirage generate -c configs/grpo_base.yaml --split dev {a}--label {name} --out {out}")
    sh(f"mirage score {out}/predictions.jsonl --spider {SPIDER_DIR} --suites {RUNS}/suites --label {name}")

## Report: accuracy table with paired bootstrap, blind-spot and training charts

In [ ]:
study = {
  "title": "Mirage: reward fidelity for GRPO text-to-SQL (Qwen2.5-Coder-0.5B, Spider) — seed 0",
  "simulated": False,
  "eval_set": "Spider dev (1,034 questions, 20 unseen databases)",
  "suites": "runs/suites",
  "arms": {"single": "runs/arms/single", "suite": "runs/arms/suite"},
  "evals": {"start": "runs/eval/start-dev", "grpo-single": "runs/eval/grpo-single-dev", "grpo-suite": "runs/eval/grpo-suite-dev"},
  "out": "runs/report",
}
import yaml
open("/kaggle/working/study.yaml", "w").write(yaml.safe_dump(study, sort_keys=False))
os.chdir("/kaggle/working")
sh("mirage report study.yaml")
print(open(f"{RUNS}/report/REPORT.md").read())